# Reproduce ParaDome's compact English classifier

CPU is sufficient; do not select a paid/GPU runtime. This notebook uses only the public, authored synthetic corpus. Never upload production messages, credentials, or private environment files.

The v1 threshold was selected on validation groups before test evaluation. This notebook reproduces that frozen version; it is not a search over the already-seen holdout. A changed model needs a new artifact version and a fresh untouched holdout.

In [ ]:
from pathlib import Path
import os, subprocess

# In Colab, clone the published repository after this lab change has landed.
# In a local notebook, open from a checkout that already contains ml/.
checkout = Path('/content/paradome-prod') if Path('/content').is_dir() else Path.cwd()
if not (checkout / 'ml/train_triage.py').exists():
    if checkout.exists() and any(checkout.iterdir()):
        raise RuntimeError('Select the checkout containing the engineering lab; do not overwrite a directory.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/husseinahmad7/paradome-prod.git', str(checkout)], check=True)
os.chdir(checkout)
assert Path('ml/train_triage.py').exists(), 'Use a revision containing the engineering lab.'
print('Public authored corpus:', Path('ml/data/triage-corpus.json').resolve())

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'ml/requirements-training.txt'], check=True)

In [ ]:
# Refit only training groups, reselect using only validation, then compare
# against the frozen artifact and report. No files are rewritten.
subprocess.run([sys.executable, 'ml/train_triage.py', '--phase', 'reproduce'], check=True)

In [ ]:
# Standard-library runtime parity, abstention, and split-integrity tests.
subprocess.run([sys.executable, '-m', 'unittest', 'engineering.test_triage', '-v'], check=True)

In [ ]:
from engineering.triage import classify, model_card
classify('The editor reports success but my message disappears after reloading.')

## A new model version

Do not tune v1 after looking at its test results. To develop v2, author a new scenario-group holdout before fitting, record its provenance, update versioned paths, and use the two-phase procedure in `ml/README.md`. Fit vocabulary and IDF on training examples only. Freeze the threshold before evaluation. Publish failures and challenge results, not just successful predictions.

The JSON artifact contains plain numeric parameters, not pickle or executable code. Production never installs these training libraries.